# Test 02: Scala/Java-Only Installation

Integration test for the `sfnb_multilang` toolkit -- Scala/Java only.

**What this tests:**
1. Toolkit installation via pip
2. JDK + Scala installation (micromamba + coursier)
3. Snowpark JAR resolution
4. Ammonite REPL initialisation
5. `%%scala` magic registration and execution
6. Python <-> Scala data transfer
7. Snowpark Scala session (credential injection)
8. Optional: Spark Connect

**Prerequisites:**
- EAI with Maven Central + GitHub + PyPI access enabled on this notebook

## 1. Install the Toolkit

In [ ]:
# Install from local source (notebooks/ is inside the package dir)
!pip install -q ..

## 2. Install Scala/Java via Toolkit

This replaces `!bash setup_scala_environment.sh`. The toolkit installs
OpenJDK 17, downloads coursier, resolves Snowpark + Scala + Ammonite JARs,
installs JPype1, and writes the metadata file.

In [ ]:
from sfnb_multilang import install

report = install(languages=["scala"])

In [ ]:
print(f"Success: {report.success}")
print(f"Env prefix: {report.env_prefix}")
print(f"Duration: {report.elapsed_seconds:.1f}s")

scala_result = report.plugin_results.get("scala")
if scala_result:
    print(f"Scala version: {scala_result.version}")
    print(f"Scala install OK: {scala_result.success}")

## 3. Verify Metadata File

The Scala plugin writes a JSON metadata file that the helper reads
for classpath configuration.

In [ ]:
import json, os

meta_path = os.path.expanduser("~/scala_jars/scala_env_metadata.json")
assert os.path.isfile(meta_path), f"FAIL: metadata not found at {meta_path}"

with open(meta_path) as f:
    meta = json.load(f)

for key in ["java_home", "scala_full_version", "snowpark_version", "ammonite_version"]:
    print(f"  {key}: {meta.get(key, 'MISSING')}")

print(f"\nPASS: Metadata file present and readable")

## 4. Set Up Scala Magic

Load the helper module and register `%%scala` / `%%java` magics.

In [ ]:
from scala_helpers import setup_scala_environment
setup_scala_environment()

## 5. Basic Scala Execution

In [ ]:
%%scala
val greeting = "Hello from Scala!"
println(greeting)
println(s"Scala version: ${scala.util.Properties.versionString}")
println(s"Java version: ${System.getProperty("java.version")}")

## 6. Python <-> Scala Variable Transfer

The `-i`/`-o` flags pass primitive values (int, float, string, bool)
between Python and Scala via JPype. For DataFrame transfer, use
Snowpark or PySpark DFs (see Section 7).

In [ ]:
py_count = 5
py_label = "test"
print(f"Sending to Scala: count={py_count}, label={py_label}")

In [ ]:
%%scala -i py_count,py_label -o scala_result
val n = py_count.asInstanceOf[Int]
val lbl = py_label.asInstanceOf[String]
val scala_result = (1 to n).map(i => s"$lbl-$i").mkString(", ")
println(s"PASS: Generated $n items: $scala_result")

## 7. Snowpark Scala Session

Test that Scala can access Snowflake via the Snowpark session
(credentials injected from the Python session).

In [ ]:
from snowflake.snowpark.context import get_active_session
from scala_helpers import bootstrap_snowpark_scala

session = get_active_session()
ok, msg = bootstrap_snowpark_scala(session)
print(f"Bootstrap: {'OK' if ok else 'FAIL'}")
if msg:
    print(msg)

In [ ]:
%%scala
val result = sfSession.sql("SELECT CURRENT_ACCOUNT() AS acct, CURRENT_ROLE() AS role").collect()
result.foreach(r => println(s"  Account: ${r.getString(0)}, Role: ${r.getString(1)}"))
println("PASS: Snowpark Scala session works")

## 8. (Optional) Java Execution

The `%%java` magic should also work since it runs on the same JVM.

In [ ]:
%%java
System.out.println("Hello from Java!");
System.out.println("Java version: " + System.getProperty("java.version"));
System.out.println("PASS: Java execution works");

## 9. (Optional) Spark Connect

Uncomment to test Spark Connect integration.

In [ ]:
# To test Spark Connect, re-install with it enabled:
# from sfnb_multilang import install
# install(config="configs/scala_only.yaml")  # edit scala_only.yaml: spark_connect.enabled: true
#
# Then test:
# %%scala
# import org.apache.spark.sql.SparkSession
# val spark = SparkSession.builder().remote("sc://localhost:15002").getOrCreate()
# spark.sql("SELECT 1").show()
# println("PASS: Spark Connect works")

## Test Summary

| Test | Expected |
|---|---|
| Toolkit install | pip succeeds |
| Scala install | report.success == True |
| Metadata | JSON file present with versions |
| Scala magic | `%%scala` cells execute |
| Data transfer | Python <-> Scala round-trip |
| Snowpark | Scala session connects to Snowflake |
| Java | `%%java` cells execute |